# Programación declarativa @ URJC
# Programación funcional
## Examen de prueba 1 (estilo convocatoria ordinaria)

La duración del examen es de 1h:30m.

# Preámbulo

In [ ]:
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

### Algunas definiciones de tipos y funciones auxiliares

In [ ]:
object Signatures:
    abstract class List[A]:

        // Common HOFs
        def foldRight[B](nil: B)(cons: (A, B) => B): B
        def foldLeft[B](initial: B)(update: (B, A) => B): B
        def map[B](f: A => B): List[B]
        def flatMap[B](f: A => List[B]): List[B]
        def filter(f: A => Boolean): List[A]
        def forall(pred: A => Boolean): Boolean
        def exists(pred: A => Boolean): Boolean

        // Common functions
        def length: Int
        def reverse: List[A]
        def drop(i: Int): List[A]
        def zip[B](other: List[B]): List[(A, B)]
        def appended(e: A): List[A]

    // List.fill(n)(e) crea una lista con n copias de e:  List.fill(3)('a') == List('a','a','a')

In [ ]:
// type Tree[A] = 1 + Tree[A] * A * Tree[A]

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:

    def void[A]: Tree[A] =
        Empty()

    def leaf[A](a: A): Node[A] =
        Node(Empty(), a, Empty())

    def right[A](a: A, tree: Tree[A]): Node[A] =
        Node(Empty(), a, tree)

    def left[A](tree: Tree[A], a: A): Node[A] =
        Node(tree, a, Empty())

    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] =
        Node(left, a, right)

import Tree._

### Definiciones auxiliares sobre la correspondencia Curry-Howard y Tipos Algebraicos de Datos

In [ ]:
trait Isomorphic[A, B]:

    def from(a: A): B

    def to(b: B): A

    // equality

    def equalA(a1: A, a2: A): Boolean =
        a1 == a2

    def equalB(b1: B, b2: B): Boolean =
        b1 == b2

    // Bijection laws

    def law1(a: A): Boolean =
        equalA(to(from(a)), a)

    def law2(b: B): Boolean =
        equalB(from(to(b)), b)

In [ ]:
type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

# Ejercicio 1 (2 puntos)

#### a) (1 punto)

Utiliza la correspondencia de Curry-Howard para demostrar la siguiente tautología de la lógica proposicional intuicionista:

$ \vdash (p \rightarrow r) \wedge (q \rightarrow r) \rightarrow p \vee q \rightarrow r$

In [ ]:
def proof[P, Q, R]: ((P => R, Q => R)) => Either[P, Q] => R =
    (f, g) => e => e match
        case Left(p)  => f(p)
        case Right(q) => g(q)

_Explicación:_ la fórmula, con paréntesis, es $((p \rightarrow r) \wedge (q \rightarrow r)) \rightarrow ((p \vee q) \rightarrow r)$. Introducimos las dos flechas con lambdas (el primer argumento es un par, que desempaquetamos en `f` y `g`) y eliminamos la disyunción con un `match`: en cada rama aplicamos la función correspondiente.

#### b) (1 punto)

Se desea utilizar la correspondencia de Curry-Howard para demostrar la validez de la siguiente tautología de la lógica clásica proposicional:

$ \vdash \neg(p \wedge q) \rightarrow \neg p \vee \neg q $

Para ello, utiliza como premisa adicional la ley del tercio excluso aplicada a la proposición $p$.

In [ ]:
def proof[P, Q](lem: ExcludedMiddle): Not[(P, Q)] => Either[Not[P], Not[Q]] =
    npq => lem[P] match
        case Left(p)   => Right(q => npq((p, q)))   // con p, construyo ¬q: si me das q, tengo (p, q) y npq lo contradice
        case Right(np) => Left(np)

# Ejercicio 2 (2 puntos)

Demuestra que, para cualquier tipo `A`, los tipos de datos `(Option[A], Boolean)` y `Either[(Boolean, A), Boolean]` son isomorfos, de las dos formas siguientes:

#### a) (0,5 puntos)

Utilizando la equivalencia entre tipos algebraicos de datos y las operaciones aritméticas.

|(Option[A], Boolean)| =

|Option[A]| * |Boolean| =

(|A| + 1) * 2 =

2 * |A| + 2 =

|Boolean| * |A| + |Boolean| =

|(Boolean, A)| + |Boolean| =

|Either[(Boolean, A), Boolean]|

#### b) (1,5 puntos)

Implementando una biyección entre ambos tipos de datos mediante una instanciación del trait `Isomorphic`.

In [ ]:
class Iso[A] extends Isomorphic[(Option[A], Boolean), Either[(Boolean, A), Boolean]]:

    def from(t: (Option[A], Boolean)): Either[(Boolean, A), Boolean] =
        t match
            case (Some(a), b) => Left((b, a))
            case (None, b)    => Right(b)

    def to(e: Either[(Boolean, A), Boolean]): (Option[A], Boolean) =
        e match
            case Left((b, a)) => (Some(a), b)
            case Right(b)     => (None, b)

In [ ]:
class TestIso(iso: Isomorphic[(Option[Int], Boolean), Either[(Boolean, Int), Boolean]])
extends AnyFlatSpec with should.Matchers:

    "law1" should "hold" in:
        iso.law1((Some(1), true)) shouldBe true
        iso.law1((Some(2), false)) shouldBe true
        iso.law1((None, true)) shouldBe true
        iso.law1((None, false)) shouldBe true

    "law2" should "hold" in:
        iso.law2(Left((true, 1))) shouldBe true
        iso.law2(Left((false, 2))) shouldBe true
        iso.law2(Right(true)) shouldBe true
        iso.law2(Right(false)) shouldBe true

In [ ]:
run(TestIso(Iso[Int]()))

# Ejercicio 3 (3 puntos)

La función `partition` recibe un predicado y una lista, y devuelve un par de listas: la primera con los elementos que cumplen el predicado y la segunda con los que no lo cumplen, respetando en ambas el orden original. El comportamiento de la función se ilustra con los siguientes casos de prueba:

In [ ]:
class TestPartition(partition: (Int => Boolean, List[Int]) => (List[Int], List[Int]))
extends AnyFlatSpec with should.Matchers:

    "partition" should "work" in:
        partition(_ % 2 == 0, List()) shouldBe (List(), List())
        partition(_ % 2 == 0, List(1, 2, 3, 4, 5)) shouldBe (List(2, 4), List(1, 3, 5))
        partition(_ > 0, List(3, 1, 2)) shouldBe (List(3, 1, 2), List())
        partition(_ > 10, List(3, 1, 2)) shouldBe (List(), List(3, 1, 2))
        partition(_ < 3, List(5, 1, 4, 2, 3)) shouldBe (List(1, 2), List(5, 4, 3))

#### a) (1 punto) Implementa la función `partition` mediante recursión (no final).

In [ ]:
def partition[A](p: A => Boolean, l: List[A]): (List[A], List[A]) =
    l match
        case Nil => (Nil, Nil)
        case head :: tail =>
            val (yes, no) = partition(p, tail)
            if p(head) then (head :: yes, no) else (yes, head :: no)

In [ ]:
run(TestPartition(partition))

#### b) (0,5 puntos) Implementa la función `partition` mediante la función `foldRight`.

In [ ]:
def partition[A](p: A => Boolean, l: List[A]): (List[A], List[A]) =
    l.foldRight((List[A](), List[A]())):
        case (head, (yes, no)) =>
            if p(head) then (head :: yes, no) else (yes, head :: no)

In [ ]:
run(TestPartition(partition))

#### c) (1 punto) Implementa la función `partition` mediante la función `foldLeft`.

In [ ]:
def partition[A](p: A => Boolean, l: List[A]): (List[A], List[A]) =
    val (yes, no) = l.foldLeft((List[A](), List[A]())):
        case ((ys, ns), e) =>
            if p(e) then (e :: ys, ns) else (ys, e :: ns)
    (yes.reverse, no.reverse)      // con foldLeft las listas se construyen al revés

In [ ]:
run(TestPartition(partition))

#### d) (0,5 puntos) Implementa la función `partition` sin recursión, utilizando exclusivamente la función de orden superior `filter`.

In [ ]:
def partition[A](p: A => Boolean, l: List[A]): (List[A], List[A]) =
    (l.filter(p), l.filter(e => !p(e)))

In [ ]:
run(TestPartition(partition))

# Ejercicio 4 (3 puntos)

Considérense las siguientes funciones `sumPairs`, `concatPairs` y `maxPairs` basadas en un mismo patrón de diseño:

In [ ]:
def sumPairs(l: List[(Int, Int)]): List[Int] =
    l match
        case Nil => Nil
        case (a, b) :: tail => (a + b) :: sumPairs(tail)

In [ ]:
def concatPairs(l: List[(String, Char)]): List[String] =
    l match
        case Nil => Nil
        case (s, c) :: tail => (s + c) :: concatPairs(tail)

In [ ]:
def maxPairs(l: List[(Int, Int)]): List[Int] =
    l match
        case Nil => Nil
        case (a, b) :: tail => (if a > b then a else b) :: maxPairs(tail)

En todas ellas, a cada par de la lista se le aplica una determinada función de dos argumentos, y se devuelve la lista con los resultados. Por ejemplo:

In [ ]:
class TestPairs(
    sumPairs: List[(Int, Int)] => List[Int],
    concatPairs: List[(String, Char)] => List[String],
    maxPairs: List[(Int, Int)] => List[Int])
extends AnyFlatSpec with should.Matchers:

    "sumPairs" should "work" in:
        sumPairs(List()) shouldBe List()
        sumPairs(List((1, 2), (3, 4), (0, 0))) shouldBe List(3, 7, 0)

    "concatPairs" should "work" in:
        concatPairs(List(("ab", 'c'), ("", 'x'))) shouldBe List("abc", "x")

    "maxPairs" should "work" in:
        maxPairs(List((1, 2), (5, 4), (3, 3))) shouldBe List(2, 5, 3)

In [ ]:
run(TestPairs(sumPairs, concatPairs, maxPairs))

#### a) (1,5 puntos) Implementa una función de orden superior, denominada `mapPairs`, que encapsule el patrón de diseño utilizado en las funciones `sumPairs`, `concatPairs` y `maxPairs`.

In [ ]:
def mapPairs[A, B, C](l: List[(A, B)])(f: (A, B) => C): List[C] =
    l match
        case Nil => Nil
        case (a, b) :: tail => f(a, b) :: mapPairs(tail)(f)

#### b) (1 punto) Implementa de una manera más modular las funciones `sumPairs`, `concatPairs` y `maxPairs` mediante la función `mapPairs`.

In [ ]:
def sumPairs(l: List[(Int, Int)]): List[Int] =
    mapPairs(l)(_ + _)

def concatPairs(l: List[(String, Char)]): List[String] =
    mapPairs(l)(_ + _)

def maxPairs(l: List[(Int, Int)]): List[Int] =
    mapPairs(l)((a, b) => if a > b then a else b)

In [ ]:
run(TestPairs(sumPairs, concatPairs, maxPairs))

#### c) (0,5 puntos) Reimplementa la función `mapPairs` mediante la función de orden superior `map`.

In [ ]:
def mapPairs[A, B, C](l: List[(A, B)])(f: (A, B) => C): List[C] =
    l.map((a, b) => f(a, b))      // o bien:  l.map { case (a, b) => f(a, b) }

In [ ]:
run(TestPairs(sumPairs, concatPairs, maxPairs))